# SKENARIO 3 — non-CoT (fine-tuned)

Run **model fine-tune non-CoT** di test set numglue + easy, ambil **semua metrik**
(pass@1/2/3 + maj@3/5). Angka CoT sudah ada dari **S2** (config identik) ->
perbandingan CoT vs non-CoT dilakukan **manual** dari kedua tabel.

Config: N=5 kandidat/soal, temp 0.7, top-p 0.95, max_new_tokens 2048 (sama persis S2). Butuh **GPU**.

**Bagi kerja 2 orang (lebih cepat):** tiap orang jalanin notebook ini di akun/sesi Kaggle sendiri,
cukup ganti `MY_SETS` di sel Config:
- Orang A: `MY_SETS = ['numglue']`
- Orang B: `MY_SETS = ['easy']`

Tiap orang hasilkan file per-set (`data/eval/s3_noncot__<set>.json`) -> gabung di sel terakhir.
**Checkpoint/resume**: generasi tiap soal dicache per dataset; run ulang nge-skip yang sudah selesai.

## 0. Setup + install peft

In [ ]:
!pip install -q -U peft transformers accelerate
# peft>=0.18 manggil dispatcher torchao; torchao bawaan Kaggle (0.10) terlalu tua -> ImportError.
# Kita tak pakai torchao -> copot saja biar peft skip jalurnya. (restart kernel setelah ini)
!pip uninstall -y -q torchao

In [ ]:
import os, sys, subprocess
from pathlib import Path
REPO = '/kaggle/working/FP_NLP'
URL  = 'https://github.com/henray404/FP_NLP.git'
if os.path.exists(REPO):
    subprocess.run(['git','-C',REPO,'fetch','-q','origin','main'], check=True)
    subprocess.run(['git','-C',REPO,'reset','--hard','origin/main'], check=True)
else:
    subprocess.run(['git','clone','-q',URL,REPO], check=True)
sys.path.insert(0, REPO); os.chdir(REPO)
print('repo ready:', REPO)

## 1. Config — adapter non-CoT, pilih dataset bagianmu, checkpoint

Add Input: dataset berisi `adapter_nocot_1.5b/`. **Ganti `MY_SETS`** ke dataset yang jadi bagianmu
(`['numglue']` atau `['easy']`; isi keduanya kalau ngerun sendiri). `CKPT_DIR` = cache resume per soal.

In [ ]:
import glob
BASE = 'Qwen/Qwen2.5-1.5B'
LABEL = 'nonCoT'

# >>> BAGI KERJA: ganti ke bagianmu. Orang A: ['numglue']  Orang B: ['easy'] <<<
MY_SETS = ['numglue', 'easy']

def find_adapter(keyword):
    hits = glob.glob(f'/kaggle/input/**/*{keyword}*/adapter_config.json', recursive=True)
    return str(Path(hits[0]).parent) if hits else None

ADAPTER_NOCOT = find_adapter('adapter_nocot')
assert ADAPTER_NOCOT, 'adapter non-CoT tak ketemu -- Add dataset adapter, atau isi ADAPTER_NOCOT manual'
print('adapter nonCoT:', ADAPTER_NOCOT)

def find_set(keyword):
    for pat in [f'/kaggle/input/**/*{keyword}*test*.jsonl', f'data/sft/test/*{keyword}*.jsonl']:
        hits = glob.glob(pat, recursive=True)
        if hits: return hits[0]
    return f'data/sft/test/{keyword}_test.jsonl'

SET_PATHS = {k: find_set(k) for k in MY_SETS}
print('bagianku:', SET_PATHS)

# checkpoint: /kaggle/working persist dalam sesi; reset --hard tak hapus untracked -> aman re-run.
CKPT_DIR = '/kaggle/working/ckpt_s3'
os.makedirs(CKPT_DIR, exist_ok=True)
print('checkpoint dir:', CKPT_DIR)

## 2. Eval bagianku + simpan per-set

In [ ]:
from src.eval.scenario_eval import load_sets
from src.eval.sample_eval import eval_specs_sampling
from src.eval.sampling_metrics import render_tables
import json

METRICS = ['pass@1', 'pass@2', 'pass@3', 'maj@3', 'maj@5']
sets = load_sets(SET_PATHS)
specs = {LABEL: {'model': BASE, 'adapter': ADAPTER_NOCOT}}

results = eval_specs_sampling(specs, sets, n_samples=5, ks_pass=(1, 2, 3), ks_maj=(3, 5),
                             temperature=0.7, top_p=0.95, max_new_tokens=2048, batch_size=16,
                             ckpt_dir=CKPT_DIR)

print('\n=== SKENARIO 3 non-CoT (bagianku) ===')
print(render_tables(results, METRICS))

Path('data/eval').mkdir(parents=True, exist_ok=True)
for s in sorted(sets):
    r = results[LABEL][s]
    line = '  '.join(f'{m}={r[m]:.3f}' for m in METRICS)
    print(f'{LABEL} {s:10} {line}  format_ok={r["format_ok_rate"]:.3f}')
    # simpan per-set biar gampang digabung dari hasil 2 orang
    out = Path(f'data/eval/s3_noncot__{s}.json')
    out.write_text(json.dumps({LABEL: {s: r}}, ensure_ascii=False, indent=2), encoding='utf-8')
    print('  ->', out)

## 3. (Opsional) Gabung hasil 2 orang -> tabel lengkap

Jalankan ini **setelah** punya `data/eval/s3_noncot__*.json` dari kedua orang
(taruh kedua file di folder `data/eval/`, misal lewat Add Input / upload). Output gabungan
`data/eval/s3_noncot.json`.

In [ ]:
import json, glob
from pathlib import Path
from src.eval.sampling_metrics import render_tables

METRICS = ['pass@1', 'pass@2', 'pass@3', 'maj@3', 'maj@5']
merged = {}
files = sorted(glob.glob('data/eval/s3_noncot__*.json'))
print('file ditemukan:', files)
for f in files:
    d = json.loads(Path(f).read_text(encoding='utf-8'))
    for label, sets_d in d.items():
        merged.setdefault(label, {}).update(sets_d)

print('\n=== SKENARIO 3 non-CoT (gabungan) ===')
print(render_tables(merged, METRICS))
Path('data/eval/s3_noncot.json').write_text(json.dumps(merged, ensure_ascii=False, indent=2), encoding='utf-8')
print('\nsummary -> data/eval/s3_noncot.json')